In [4]:
from src.data_loader import load_and_merge
from src.preprocessing import Preprocessor

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

pre = Preprocessor()
df = pre.encode_ids(df)

user_item = pre.create_user_item_matrix(df)

print(user_item.shape)

(611, 32918)


Matrix Factorization Training

In [35]:
from models.matrix_factorization import MatrixFactorization
from src.preprocessing import Preprocessor
from src.data_loader import load_and_merge

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

pre = Preprocessor()
df = pre.encode_ids(df)

n_users, n_items = pre.get_num_users_items(df)

mf = MatrixFactorization(
    n_users,
    n_items,
    n_factors=40,
    epochs=20,
    lr=0.005,
    reg=0.05
)

mf.fit(df)

print("Prediction (user 0, item 0):", mf.predict(0, 0))

Epoch 1/20, Loss: 124432.34
Epoch 2/20, Loss: 124386.22
Epoch 3/20, Loss: 124344.76
Epoch 4/20, Loss: 124301.77
Epoch 5/20, Loss: 124251.18
Epoch 6/20, Loss: 124184.43
Epoch 7/20, Loss: 124087.32
Epoch 8/20, Loss: 123934.95
Epoch 9/20, Loss: 123683.23
Epoch 10/20, Loss: 123256.30
Epoch 11/20, Loss: 122532.26
Epoch 12/20, Loss: 121342.07
Epoch 13/20, Loss: 119516.94
Epoch 14/20, Loss: 117014.49
Epoch 15/20, Loss: 114043.46
Epoch 16/20, Loss: 110984.17
Epoch 17/20, Loss: 108117.79
Epoch 18/20, Loss: 105494.37
Epoch 19/20, Loss: 103035.66
Epoch 20/20, Loss: 100662.95
Prediction (user 0, item 0): 4.225239390620893


In [37]:
import pickle

with open("./outputs/saved_models/mf.pkl", "wb") as f:
    pickle.dump(mf, f)

Hybrid Predictor (MF + Genre Similarity)

In [ ]:
import pandas as pd
from models.hybrid import HybridRecommender

movies = pd.read_csv("./data/movies.csv")

hybrid = HybridRecommender(mf, movies, alpha=0.7)

user = 0

user_history = df[df['user'] == user]['movieId'].values[:10]

item = 0  

item_movieId = pre.movie_encoder.inverse_transform([item])[0]

print(hybrid.predict(user, item_movieId, user_history))

2.6764395172431303


Evaluation


In [ ]:
from evaluation.evaluate import evaluate_models
from models.matrix_factorization import MatrixFactorization
from src.preprocessing import Preprocessor
from src.data_loader import load_and_merge

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

df_small = df.sample(20000, random_state=42)

pre_small = Preprocessor()
df_small = pre_small.encode_ids(df_small)

results = evaluate_models(df_small, pre_small)

print(results)

Epoch 1/5, Loss: 14408.33
Epoch 2/5, Loss: 12940.86
Epoch 3/5, Loss: 12169.48
Epoch 4/5, Loss: 11645.53
Epoch 5/5, Loss: 11247.49
Starting SVD++...
Epoch 1, Loss: 14407.86
Epoch 2, Loss: 12899.59
Finished SVD++
                 model      RMSE       MAE
0             Baseline  0.984017  0.754374
1  MatrixFactorization  0.911760  0.718673
2                SVD++  0.931997  0.742922
